In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DWI_ZIP_PATH  = "/content/drive/MyDrive/datasets/DWI_NIFTI.zip"
MASK_ZIP_PATH = "/content/drive/MyDrive/datasets/MASK_NIFTI.zip"

In [3]:
import os, zipfile, glob

def safe_unzip(zip_path, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target_dir)

    # Recursively search .nii and .nii.gz BUT ignore the extracted_nii folder
    nii_files = []
    for root, dirs, files in os.walk(target_dir):

        # Skip duplicate folder
        if "extracted_nii" in root:
            continue

        for f in files:
            if f.endswith(".nii") or f.endswith(".nii.gz"):
                nii_files.append(os.path.join(root, f))

    return sorted(nii_files)

dwi_list = safe_unzip(DWI_ZIP_PATH, "/content/DWI_NIFTI_extracted")
mask_list = safe_unzip(MASK_ZIP_PATH, "/content/MASK_NIFTI_extracted")

print(f"# DWI volumes found: {len(dwi_list)}")
print(f"# MASK volumes found: {len(mask_list)}")
print("DWI examples:", dwi_list[:5])
print("MASK examples:", mask_list[:5])


# DWI volumes found: 250
# MASK volumes found: 250
DWI examples: ['/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0001_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0002_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0003_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0004_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0005_dwi.nii.gz']
MASK examples: ['/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0001_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0002_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0003_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0004_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0005_mask.nii.gz']


In [4]:
# Pairing rule
def pair_volumes(dwi_files, mask_files):
    def key(f):
        base = os.path.basename(f)
        base = os.path.splitext(base)[0]
        base = base.replace("_mask", "")
        base = base.replace("-mask", "")
        base = base.replace("_dwi", "")
        base = base.replace("-dwi", "")
        return base

    dwi_map = {key(f): f for f in dwi_files}
    mask_map = {key(f): f for f in mask_files}

    pairs = []
    for k, dwi_path in dwi_map.items():
        if k in mask_map:
            pairs.append((dwi_path, mask_map[k]))
    return pairs

pairs = pair_volumes(dwi_list, mask_list)
print("Paired volumes:", len(pairs))
pairs[:5]



Paired volumes: 250


[('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0001_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0001_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0002_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0002_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0003_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0003_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0004_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0004_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0005_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0005_mask.nii.gz')]

In [6]:
!pip install monai[all] -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.

In [7]:
import os
import random
from glob import glob

import nibabel as nib
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from monai.networks.nets import resnet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric

from torch.optim import Adam
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# Fix seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda


In [8]:
# Fixed slice size — can adjust based on memory/GPU
TARGET_SIZE = (192, 192)

class DWISliceDataset(Dataset):
    """Extract 2D axial slices from 3D DWI + Mask volumes, resized to TARGET_SIZE."""

    def __init__(self, pairs, target_size=(192,192), keep_empty_slices=False, transform=None):
        self.items = []
        self.pairs = pairs
        self.transform = transform
        self.target_size = target_size
        self.keep_empty_slices = keep_empty_slices

        print("Preparing dataset…")
        for dwi_path, mask_path in pairs:
            try:
                dwi_data = nib.load(dwi_path).get_fdata(dtype=np.float32)
                mask_data = nib.load(mask_path).get_fdata(dtype=np.float32)
            except Exception:
                print("Skipping unreadable:", dwi_path)
                continue

            # Align shapes
            H = min(dwi_data.shape[0], mask_data.shape[0])
            W = min(dwi_data.shape[1], mask_data.shape[1])
            S = min(dwi_data.shape[2], mask_data.shape[2])
            dwi_data = dwi_data[:H, :W, :S]
            mask_data = mask_data[:H, :W, :S]

            for z in range(S):
                ms = mask_data[:, :, z]
                if (not keep_empty_slices) and np.all(ms == 0):
                    continue
                self.items.append((dwi_path, mask_path, z))

        print(f"✔ Prepared {len(self.items)} slices total")

    def _resize(self, t):
        # t shape: (1,H,W)
        t = t.unsqueeze(0)  # => (1,1,H,W)
        t = F.interpolate(t, size=self.target_size, mode="bilinear", align_corners=False)
        return t.squeeze(0)  # => (1,H,W)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        dwi_path, mask_path, z = self.items[idx]

        dwi = nib.load(dwi_path).get_fdata(dtype=np.float32)[:, :, z]
        mask = nib.load(mask_path).get_fdata(dtype=np.float32)[:, :, z]

        mask = (mask > 0).astype(np.float32)

        # z-score normalization
        mean, std = dwi.mean(), dwi.std()
        if std == 0: std = 1
        dwi = (dwi - mean) / std

        # Convert to tensors and add channel dim
        dwi = torch.from_numpy(dwi).unsqueeze(0).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        # Resize slices
        dwi = self._resize(dwi)
        mask = self._resize(mask)
        mask = (mask > 0.5).float()  # re-binarize

        if self.transform:
            dwi, mask = self.transform(dwi, mask)

        return dwi, mask


In [9]:
# Prepare pairs if not defined
try:
    pairs
except NameError:
    print("`pairs` not found, scanning /content...")
    dwi_list = sorted(glob('/content/**/*.nii*', recursive=True))
    mask_list = []  # user must match masks
    pairs = []  # manually set or regenerate
    print("Please set `pairs` manually or rerun pairing code.")

# Parameters
BATCH_SIZE = 4
VAL_SPLIT = 0.15
NUM_WORKERS = 2

# Create dataset
dataset = DWISliceDataset(pairs, target_size=TARGET_SIZE, keep_empty_slices=False)

# Split train/val
val_count = max(1, int(len(dataset) * VAL_SPLIT))
train_count = len(dataset) - val_count
train_ds, val_ds = random_split(dataset, [train_count, val_count])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Quick check
imgs, masks = next(iter(train_loader))
print("Batch shapes:", imgs.shape, masks.shape)  # should be [B,1,H,W]


Preparing dataset…
✔ Prepared 4827 slices total
Batch shapes: torch.Size([4, 1, 192, 192]) torch.Size([4, 1, 192, 192])


In [11]:
# ============================
# CELL 4 — Model (Working)
# ============================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

# Get ResNet50 backbone (pretrained=False)
resnet = resnet50(weights=None)

# Adjust first conv for 1-channel input
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Extract feature layers only (exclude avgpool and fc)
backbone = nn.Sequential(
    resnet.conv1,
    resnet.bn1,
    resnet.relu,
    resnet.maxpool,
    resnet.layer1,
    resnet.layer2,
    resnet.layer3,
    resnet.layer4
)  # output: [B, 2048, H/32, W/32]

class SimpleDecoderSeg(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.decoder = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, x):
        feats = self.backbone(x)  # [B, 2048, H/32, W/32]
        out = self.decoder(feats)
        # Resize to input H,W
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleDecoderSeg(backbone).to(device)

# Quick forward test
x = torch.randn(2, 1, 192, 192).to(device)
y = model(x)
print("Input shape:", x.shape, "Output shape:", y.shape)


Input shape: torch.Size([2, 1, 192, 192]) Output shape: torch.Size([2, 1, 192, 192])


In [12]:
# Loss functions
bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss(sigmoid=True)

# Optimizer
optimizer = Adam(model.parameters(), lr=1e-4)

# Metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")
hausdorff_metric = HausdorffDistanceMetric(include_background=False, reduction="mean")

# IoU helper
def compute_iou(pred_mask: torch.Tensor, true_mask: torch.Tensor, thr: float = 0.5, eps=1e-6):
    pred = (torch.sigmoid(pred_mask) > thr).float()
    true = (true_mask > 0.5).float()
    inter = (pred * true).sum(dim=[1,2,3])
    union = (pred + true - pred * true).sum(dim=[1,2,3])
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()


In [13]:
NUM_EPOCHS = 20
CHECKPOINT_DIR = '/content/checkpoints'
CHECKPOINT_EVERY = 5
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

writer = SummaryWriter(log_dir='/content/runs/resnet50_monai')
best_val_iou = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = bce_loss(preds, masks) + dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        pbar.set_postfix({'loss': f"{train_loss_sum / (pbar.n + 1):.4f}"})

    avg_train_loss = train_loss_sum / max(1, len(train_loader))
    writer.add_scalar('train/loss', avg_train_loss, epoch)

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_iou_list = []
    val_dice_list = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)

            val_loss = bce_loss(preds, masks).item() + dice_loss(preds, masks).item()
            val_loss_sum += val_loss

            # Binarize predictions
            pred_bin = (torch.sigmoid(preds) > 0.5).float()

            # Compute Dice manually per batch
            inter = (pred_bin * masks).sum(dim=[1,2,3])
            union = pred_bin.sum(dim=[1,2,3]) + masks.sum(dim=[1,2,3])
            batch_dice = (2 * inter / (union + 1e-6)).mean().item()
            val_dice_list.append(batch_dice)

            val_iou_list.append(compute_iou(preds, masks))

    avg_val_loss = val_loss_sum / max(1, len(val_loader))
    avg_val_iou = float(np.mean(val_iou_list))
    avg_val_dice = float(np.mean(val_dice_list))

    writer.add_scalar('val/loss', avg_val_loss, epoch)
    writer.add_scalar('val/iou', avg_val_iou, epoch)
    writer.add_scalar('val/dice', avg_val_dice, epoch)

    # Print metrics including Dice
    print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, "
          f"val_loss={avg_val_loss:.4f}, val_iou={avg_val_iou:.4f}, val_dice={avg_val_dice:.4f}")

    # Save best model
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou': avg_val_iou,
            'val_dice': avg_val_dice
        }, os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
        print("Saved new best model!")

    # Periodic checkpoint
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pth'))

writer.close()


Epoch 1/20 [Val]: 100%|██████████| 181/181 [00:18<00:00,  9.62it/s]


Epoch 1: train_loss=1.2485, val_loss=1.0676, val_iou=0.1387, val_dice=0.1967
Saved new best model!


Epoch 2/20 [Val]: 100%|██████████| 181/181 [00:18<00:00,  9.85it/s]


Epoch 2: train_loss=0.9831, val_loss=0.9034, val_iou=0.2361, val_dice=0.3181
Saved new best model!


Epoch 3/20 [Val]: 100%|██████████| 181/181 [00:18<00:00,  9.69it/s]


...

Epoch 18: train_loss=0.3956, val_loss=0.4691, val_iou=0.4449, val_dice=0.5635


Epoch 19/20 [Val]: 100%|██████████| 181/181 [00:17<00:00, 10.11it/s]


Epoch 19: train_loss=0.3915, val_loss=0.4505, val_iou=0.4635, val_dice=0.5824
Saved new best model!


Epoch 20/20 [Val]: 100%|██████████| 181/181 [00:18<00:00,  9.69it/s]


Epoch 20: train_loss=0.3872, val_loss=0.4715, val_iou=0.4421, val_dice=0.5618
